In [1]:
import json
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from collections import defaultdict, Counter

# Загрузка данных из файла
try:
    with open('C:/Users/above/Мой диск/results.json', 'r', encoding='utf-8') as f:
        data = json.load(f)
except FileNotFoundError:
    print("Ошибка: Файл results.json не найден.")
    exit()

# 1. Подготовка данных

# Создаем словарь: frame_idx -> set(person_ids) из раздела "raw_poses"
poses_by_frame = defaultdict(set)
for pose in data['raw_poses']:
    # Проверяем, что person_id и frame_idx существуют
    if 'frame_idx' in pose and 'person_id' in pose:
        poses_by_frame[pose['frame_idx']].add(pose['person_id'])

# Создаем список действий из раздела "pose_actions"
# Структура: (frame_idx, person_id, action_name)
actions_data = []
for action in data['pose_actions']:
    if 'frame_idx' in action and 'person_id' in action and action.get('action', {}).get('action_name'):
        actions_data.append((
            action['frame_idx'],
            action['person_id'],
            action['action']['action_name']
        ))

# Сортируем действия по кадру для правильной последовательности
actions_data.sort(key=lambda x: x[0])

# 2. Анализ частоты действий (общее количество вхождений) 
action_counter = Counter([action_name for _, _, action_name in actions_data])

print("Анализ частоты действий")
print(f"Всего записей о действиях: {len(actions_data)}")
print("\nЧастота каждого действия (количество раз, когда оно было обнаружено):")
for action, count in action_counter.most_common():
    print(f"  - {action}: {count}")

# Построение графика частоты действий
if action_counter:
    plt.figure(figsize=(12, 6))
    actions = list(action_counter.keys())
    counts = list(action_counter.values())
    plt.barh(actions, counts, color='skyblue')
    plt.xlabel('Количество вхождений')
    plt.ylabel('Действие')
    plt.title('Частота обнаруженных действий')
    plt.tight_layout()
    plt.show()
else:
    print("Нет данных о действиях для построения графика частоты.")


# 3. Распределение действий на оси времени (по кадрам) 
# Определяем временные интервалы для каждого актера и его действий
# Алгоритм: проходим по отсортированному списку действий для каждого актера,
# ищем, где оно заканчивается (следующее действие того же актера или исчезновение из поз).

# Группируем действия по person_id
actions_by_person = defaultdict(list)
for frame, person, action in actions_data:
    actions_by_person[person].append((frame, action))

# Словарь для хранения временных интервалов действий: (person_id, action_name) -> [(start_frame, end_frame), ...]
action_intervals_by_person = defaultdict(list)

for person_id, person_actions in actions_by_person.items():
    # Сортируем действия этого актера по кадру
    person_actions.sort(key=lambda x: x[0])
    # Получаем кадры, где этот актер присутствует в позах
    person_pose_frames = sorted([frame for frame, persons in poses_by_frame.items() if person_id in persons])
    if not person_pose_frames:
        continue

    # Обрабатываем каждое действие актера
    for i, (start_frame, action_name) in enumerate(person_actions):
        # Определяем кадр, на котором действие заканчивается
        # Начальное предположение - до бесконечности или до последнего кадра его присутствия
        end_frame = person_pose_frames[-1]
        
        # 1. Если это не последнее действие актера, то конец - это кадр перед началом следующего действия
        if i + 1 < len(person_actions):
            next_action_frame = person_actions[i+1][0]
            # Ищем ближайший кадр, где актер еще присутствует, перед следующим действием
            # Конец интервала - это последний кадр, где он есть, но который меньше, чем начало следующего действия
            possible_end_frames = [f for f in person_pose_frames if f < next_action_frame]
            if possible_end_frames:
                end_frame = max(possible_end_frames)
            else:
                # Если нет кадров до следующего действия, значит действие было на одном кадре
                end_frame = start_frame
        else:
            # 2. Для последнего действия - конец, когда актер перестает обнаруживаться
            # Ищем последний кадр, где актер присутствует
            end_frame = max(person_pose_frames)
        
        # Сохраняем интервал, если он имеет ненулевую длительность
        if end_frame >= start_frame:
            action_intervals_by_person[(person_id, action_name)].append((start_frame, end_frame))

# Визуализация распределения действий по времени для каждого актера
if action_intervals_by_person:
    # Группируем по актеру для отображения на разных графиках
    actors = sorted(list(set([pid for pid, _ in action_intervals_by_person.keys()])))
    num_actors = len(actors)
    
    # Если актеров много, покажем первые 5 или настроим отображение
    if num_actors > 10:
        print(f"Найдено {num_actors} актеров. Для визуализации будут показаны первые 10.")
        actors_to_plot = actors[:10]
    else:
        actors_to_plot = actors
        
    fig, axes = plt.subplots(len(actors_to_plot), 1, figsize=(12, 3 * len(actors_to_plot)), sharex=True)
    if len(actors_to_plot) == 1:
        axes = [axes]
    
    # Находим общий диапазон кадров для всех данных
    all_frames = []
    for (pid, _), intervals in action_intervals_by_person.items():
        for start, end in intervals:
            all_frames.extend([start, end])
    if all_frames:
        global_min_frame = min(all_frames)
        global_max_frame = max(all_frames)
    else:
        global_min_frame, global_max_frame = 0, 1
        
    for i, actor_id in enumerate(actors_to_plot):
        ax = axes[i]
        # Получаем все действия этого актера
        actor_intervals = {action_name: intervals for (pid, action_name), intervals in action_intervals_by_person.items() if pid == actor_id}
        
        # Создаем цветовую карту для действий этого актера
        unique_actions = list(actor_intervals.keys())
        colors = plt.cm.tab10(np.linspace(0, 1, len(unique_actions)))
        color_map = {action: color for action, color in zip(unique_actions, colors)}
        
        y_position = 1
        for action_name, intervals in actor_intervals.items():
            color = color_map[action_name]
            for start, end in intervals:
                ax.barh(y_position, end - start, left=start, height=0.8, color=color, alpha=0.7, label=action_name if y_position == 1 else "")
            y_position += 1
        
        ax.set_title(f'Актер {actor_id}')
        ax.set_ylabel('Действия')
        ax.set_yticks(range(1, y_position))
        ax.set_yticklabels(unique_actions)
        ax.set_xlim(global_min_frame, global_max_frame)
        ax.grid(axis='x', linestyle='--', alpha=0.7)
        
        # Добавляем легенду только для первого графика, чтобы не загромождать
        if i == 0:
            handles = [plt.Rectangle((0,0),1,1, color=color_map[act]) for act in unique_actions]
            ax.legend(handles, unique_actions, loc='upper right', fontsize='small')
    
    axes[-1].set_xlabel('Номер кадра')
    plt.suptitle('Распределение действий по времени для каждого актера', fontsize=16)
    plt.tight_layout()
    plt.show()
else:
    print("Нет данных для построения распределения действий по времени.")

# 4. Анализ количества актеров в кадре 
# Используем словарь poses_by_frame
frames_with_poses = sorted(poses_by_frame.keys())
if frames_with_poses:
    actors_count_by_frame = {frame: len(poses_by_frame[frame]) for frame in frames_with_poses}
    
    # Выводим статистику
    print("\n Анализ количества актеров в кадре")
    print(f"Всего кадров с обнаруженными позами: {len(actors_count_by_frame)}")
    avg_actors = sum(actors_count_by_frame.values()) / len(actors_count_by_frame)
    print(f"Среднее количество актеров в кадре: {avg_actors:.2f}")
    print(f"Максимальное количество актеров в кадре: {max(actors_count_by_frame.values())}")
    print(f"Минимальное количество актеров в кадре: {min(actors_count_by_frame.values())}")
    
    # Визуализация количества актеров в кадре по времени
    plt.figure(figsize=(15, 6))
    frames = list(actors_count_by_frame.keys())
    counts = list(actors_count_by_frame.values())
    plt.plot(frames, counts, linewidth=0.5, color='green', alpha=0.7)
    plt.fill_between(frames, counts, alpha=0.2, color='green')
    plt.xlabel('Номер кадра')
    plt.ylabel('Количество актеров')
    plt.title('Количество актеров в кадре (на основе данных о позах)')
    plt.grid(axis='both', linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.show()
else:
    print("Нет данных о позах для анализа количества актеров в кадре.")

Ошибка: Файл results.json не найден.


NameError: name 'data' is not defined